# Базовое моделирование на подготовленном датасете

## Цель этапа

Этот ноутбук добавляет воспроизводимый базовый эксперимент после первичного анализа: регрессию на `final_dataset_for_modeling.csv`. Цель этапа — проверить, есть ли в подготовленных признаках первичный предсказательный сигнал для интенсивности изменения береговой бровки.

Это не финальная нейросетевая часть и не причинная экологическая интерпретация. Базовый эксперимент нужен как точка отсчёта перед сравнением более сложных статистических и нейросетевых методов.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from src.analysis.baseline_modeling import (
    TARGET_DECISION_NOTE,
    build_feature_frame,
    choose_target,
    run_baseline_modeling,
    build_models,
)

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 100)

project_root = Path.cwd()
if not (project_root / 'data').exists():
    project_root = project_root.parent
processed_dir = project_root / 'data' / 'processed'
reports_dir = project_root / 'reports'


## Загрузка данных

Основной вход — `data/processed/final_dataset_for_modeling.csv`. Это компактный производный датасет, собранный из безопасного аналитического слоя первого этапа.

Ноутбук запускает тот же воспроизводимый модуль, что и команда `python -m src.analysis.baseline_modeling`, а затем читает сохранённые таблицы и графики из `reports/`.

In [ ]:
outputs = run_baseline_modeling(verbose=False)
pd.DataFrame({
    'показатель': [
        'целевая переменная',
        'число строк',
        'число признаков до кодирования категорий',
        'модель с минимальной MAE на тестовой выборке',
        'MAE этой модели',
        'RMSE этой модели',
        'R2 этой модели',
    ],
    'значение': [
        outputs['target'],
        outputs['n_rows'],
        outputs['n_features'],
        outputs['best_model'],
        round(outputs['best_test_mae'], 4),
        round(outputs['best_test_rmse'], 4),
        round(outputs['best_test_r2'], 4),
    ],
})

In [ ]:
data = pd.read_csv(processed_dir / 'final_dataset_for_modeling.csv')
metrics = pd.read_csv(reports_dir / 'tables' / 'baseline_modeling_metrics.csv')
group_metrics = pd.read_csv(reports_dir / 'tables' / 'baseline_group_validation_metrics.csv')
feature_importance = pd.read_csv(reports_dir / 'tables' / 'baseline_feature_importance.csv')
worst_predictions = pd.read_csv(reports_dir / 'tables' / 'baseline_worst_predictions.csv')

target = choose_target(data)
X, y, metadata, numeric_features, categorical_features, excluded_columns = build_feature_frame(data, target)

pd.DataFrame({
    'показатель': [
        'строк в исходном производном датасете',
        'строк с непустой целевой переменной',
        'числовых признаков',
        'категориальных признаков',
        'всего признаков до кодирования категорий',
    ],
    'значение': [len(data), len(y), len(numeric_features), len(categorical_features), X.shape[1]],
})

## Целевая переменная и признаки

Целевая переменная выбирается в модуле явно и воспроизводимо: `retreat_rate_abs_m_per_year`. Это беззнаковая интенсивность изменения береговой бровки в м/год. Она выбрана потому, что нормирована на длительность интервала и не требует автоматической трактовки знака смещения.

Из признаков исключаются сама целевая переменная, другие прямые метрики береговой бровки (`retreat_m`, `retreat_rate_m_per_year`, `retreat_abs_m`), длительность интервала как компонент формулы целевой переменной, идентификаторы, даты в исходном текстовом виде и служебные тексты проверки качества. Даты используются только в виде осторожного признака `interval_mid_year`, который описывает положение интервала во времени.

In [ ]:
print(TARGET_DECISION_NOTE)
display(y.describe().rename('target_summary').to_frame())

feature_table = pd.DataFrame({
    'feature_name': numeric_features + categorical_features,
    'feature_type': ['числовой'] * len(numeric_features) + ['категориальный'] * len(categorical_features),
})
feature_table

## Методика проверки качества

Используется фиксированный `RANDOM_STATE = 42`. Основная таблица сохраняет случайное разбиение на обучающую и тестовую выборки в пропорции 80/20. Кросс-валидация считается только на обучающей выборке.

Дополнительно считается групповая проверка с разделением по профилям, если доступна колонка `profile_id`. В этом случае тестовая выборка содержит профили, которые не попадали в обучение. Если `profile_id` недоступен, модуль пробует `site_id`; если групповых колонок нет, это явно записывается в отчёте.

Предобработка находится внутри независимой цепочки `sklearn Pipeline` для каждой модели: числовые признаки заполняются медианой и масштабируются, категориальные признаки заполняются наиболее частым значением и кодируются. Это снижает риск утечки информации из тестовой выборки.

## Сравнение моделей

Сравниваются пять подходов:

- `DummyRegressor` нужен как нижняя точка отсчёта. Он почти ничего не изучает и показывает базовый уровень качества;
- `Ridge` используется как интерпретируемая линейная модель;
- `RandomForestRegressor` используется как нелинейная ансамблевая модель;
- `HistGradientBoostingRegressor` используется как более сильная табличная модель;
- `MLPRegressor` используется как нейросетевая регрессионная модель.

Если `MLPRegressor` не оказывается лучшей моделью, это не отменяет нейросетевую часть работы. Это означает, что на данном объёме и качестве данных простая нейросеть не получила преимущества над табличными ансамблевыми моделями.


In [ ]:
metrics_display = metrics[[
    'validation_scheme',
    'model',
    'cv_mae_mean',
    'cv_rmse_mean',
    'cv_r2_mean',
    'test_mae',
    'test_rmse',
    'test_r2',
    'is_best_by_test_mae',
]].copy()
for column in ['cv_mae_mean', 'cv_rmse_mean', 'cv_r2_mean', 'test_mae', 'test_rmse', 'test_r2']:
    metrics_display[column] = metrics_display[column].round(4)
metrics_display

## График сравнения моделей

На графике ниже MAE и RMSE показывают ошибку в единицах целевой переменной, то есть в м/год. R2 показывает, насколько модель лучше или хуже простого прогноза средним значением на тестовой выборке.

Формулировка «лучшая модель» здесь означает только «лучшая по MAE на тестовой выборке при данном случайном разбиении». Кросс-валидация и групповая проверка могут давать другую картину, поэтому базовый эксперимент следует читать как первичную оценку предсказательного сигнала, а не как финальный вывод.

In [ ]:
display(Image(filename=str(reports_dir / 'figures' / 'baseline_modeling_metrics.png')))
display(Image(filename=str(reports_dir / 'figures' / '03_model_comparison_mae.png')))

## Детальная проверка нейросетевой модели

В этом блоке отдельно рассматривается нейросетевая модель `MLPRegressor`.

Из-за сильной асимметрии целевой переменной дополнительно используется вариант с логарифмическим преобразованием `log(1 + y)`. Такой вариант не меняет исходные данные, но снижает влияние редких экстремальных значений на обучение и визуализацию результата.

Основной график ниже показывает не отдельные точки, а агрегированный временной ряд: фактические значения и прогноз нейросетевой модели по последней временной части выборки.


In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np

from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore", category=ConvergenceWarning)

X_train, X_test, y_train, y_test, metadata_train, metadata_test = train_test_split(
    X,
    y,
    metadata,
    test_size=0.2,
    random_state=42,
)

models = build_models(numeric_features, categorical_features)

if "MLPRegressor" not in models:
    raise ValueError("В build_models не найдена модель 'MLPRegressor'")

mlp = clone(models["MLPRegressor"])
mlp.fit(X_train, y_train)
mlp_pred = mlp.predict(X_test)
mlp_pred = np.clip(mlp_pred, 0, None)

log_mlp = TransformedTargetRegressor(
    regressor=clone(models["MLPRegressor"]),
    func=np.log1p,
    inverse_func=np.expm1,
)

log_mlp.fit(X_train, y_train)
log_mlp_pred = log_mlp.predict(X_test)
log_mlp_pred = np.clip(log_mlp_pred, 0, None)

def regression_metrics(name, prediction):
    return {
        "model": name,
        "test_mae": mean_absolute_error(y_test, prediction),
        "test_rmse": np.sqrt(mean_squared_error(y_test, prediction)),
        "test_r2": r2_score(y_test, prediction),
        "test_mae_log_scale": mean_absolute_error(np.log1p(y_test), np.log1p(prediction)),
    }

neural_metrics = pd.DataFrame(
    [
        regression_metrics("MLPRegressor", mlp_pred),
        regression_metrics("MLPRegressor_log_target", log_mlp_pred),
    ]
)

for column in ["test_mae", "test_rmse", "test_r2", "test_mae_log_scale"]:
    neural_metrics[column] = neural_metrics[column].round(4)

neural_metrics_path = reports_dir / "tables" / "03_neural_mlp_prediction_metrics.csv"
neural_metrics.to_csv(neural_metrics_path, index=False)

log_predictions = metadata_test.reset_index(drop=True).copy()
log_predictions["actual"] = np.asarray(y_test)
log_predictions["predicted_log_mlp"] = log_mlp_pred
log_predictions["absolute_error"] = np.abs(log_predictions["actual"] - log_predictions["predicted_log_mlp"])

log_predictions_path = reports_dir / "tables" / "03_log_mlp_test_predictions.csv"
log_predictions.to_csv(log_predictions_path, index=False)

display(neural_metrics)
print("Сохранено:", neural_metrics_path)
print("Сохранено:", log_predictions_path)


### Временное сопоставление факта и прогноза

Для наглядной демонстрации нейросетевой модели ниже строится график, похожий на прогнозный временной ряд.

Наблюдения упорядочиваются по времени. Модель обучается на более ранней части данных, а прогноз строится для последней временной части выборки. Чтобы график не разрушался отдельными экстремальными наблюдениями, значения агрегируются по годам с помощью медианы.

Чёрная линия показывает фактическую медианную интенсивность изменения береговой бровки по годам. Пунктирная линия показывает медианный прогноз нейросетевой модели для последней временной части данных. Голубая область обозначает период, на котором строился прогноз.


In [ ]:
temporal_frame = metadata.reset_index(drop=True).copy()
temporal_frame["_row_id"] = np.arange(len(temporal_frame))
temporal_frame["actual"] = np.asarray(y)

if "date_end" in temporal_frame.columns:
    temporal_dates = pd.to_datetime(temporal_frame["date_end"], errors="coerce")
else:
    temporal_dates = pd.Series(pd.NaT, index=temporal_frame.index)

if temporal_dates.notna().sum() < 10 and "interval_id" in temporal_frame.columns:
    extracted_dates = temporal_frame["interval_id"].astype(str).str.extract(r"(\d{4}-\d{2}-\d{2})$")[0]
    temporal_dates = pd.to_datetime(extracted_dates, errors="coerce")

temporal_frame["plot_year"] = temporal_dates.dt.year.astype("float")

if temporal_frame["plot_year"].notna().sum() < 10:
    if "interval_mid_year" in data.columns:
        candidate_years = pd.to_numeric(data["interval_mid_year"], errors="coerce")
        temporal_frame["plot_year"] = candidate_years.reset_index(drop=True).reindex(temporal_frame.index).to_numpy()
    else:
        raise ValueError("Не удалось получить временную ось из date_end, interval_id или interval_mid_year.")

temporal_frame = temporal_frame.dropna(subset=["plot_year", "actual"]).sort_values(["plot_year", "_row_id"]).reset_index(drop=True)

split_position = int(len(temporal_frame) * 0.8)
split_position = min(max(split_position, 1), len(temporal_frame) - 1)

train_ids = temporal_frame.iloc[:split_position]["_row_id"].to_numpy()
test_ids = temporal_frame.iloc[split_position:]["_row_id"].to_numpy()

def take_rows(obj, row_ids):
    if hasattr(obj, "iloc"):
        return obj.iloc[row_ids]
    return obj[row_ids]

X_train_time = take_rows(X, train_ids)
X_test_time = take_rows(X, test_ids)
y_train_time = take_rows(y, train_ids)
y_test_time = take_rows(y, test_ids)

temporal_log_mlp = TransformedTargetRegressor(
    regressor=clone(models["MLPRegressor"]),
    func=np.log1p,
    inverse_func=np.expm1,
)

temporal_log_mlp.fit(X_train_time, y_train_time)
temporal_pred = temporal_log_mlp.predict(X_test_time)
temporal_pred = np.clip(temporal_pred, 0, None)

temporal_test = temporal_frame.set_index("_row_id").loc[test_ids].reset_index()
temporal_test["predicted_log_mlp"] = temporal_pred
temporal_test["absolute_error"] = np.abs(temporal_test["actual"] - temporal_test["predicted_log_mlp"])

temporal_mae = mean_absolute_error(y_test_time, temporal_pred)
temporal_rmse = np.sqrt(mean_squared_error(y_test_time, temporal_pred))
temporal_r2 = r2_score(y_test_time, temporal_pred)

temporal_metrics = pd.DataFrame(
    [
        {
            "model": "MLPRegressor_log_target",
            "split": "последняя временная часть выборки",
            "train_rows": len(train_ids),
            "test_rows": len(test_ids),
            "test_mae": temporal_mae,
            "test_rmse": temporal_rmse,
            "test_r2": temporal_r2,
        }
    ]
)

for column in ["test_mae", "test_rmse", "test_r2"]:
    temporal_metrics[column] = temporal_metrics[column].round(4)

temporal_metrics_path = reports_dir / "tables" / "03_temporal_log_mlp_metrics.csv"
temporal_predictions_path = reports_dir / "tables" / "03_temporal_log_mlp_predictions.csv"

temporal_metrics.to_csv(temporal_metrics_path, index=False)
temporal_test.to_csv(temporal_predictions_path, index=False)

display(temporal_metrics)

temporal_frame["plot_year_int"] = temporal_frame["plot_year"].round().astype(int)
temporal_test["plot_year_int"] = temporal_test["plot_year"].round().astype(int)

actual_by_year = (
    temporal_frame
    .groupby("plot_year_int", observed=True)
    .agg(
        actual_median=("actual", "median"),
        actual_mean=("actual", "mean"),
        n_actual=("actual", "size"),
    )
    .reset_index()
)

predicted_by_year = (
    temporal_test
    .groupby("plot_year_int", observed=True)
    .agg(
        predicted_median=("predicted_log_mlp", "median"),
        predicted_mean=("predicted_log_mlp", "mean"),
        test_actual_median=("actual", "median"),
        n_predicted=("actual", "size"),
    )
    .reset_index()
)

temporal_plot = actual_by_year.merge(predicted_by_year, on="plot_year_int", how="left")

fig, ax = plt.subplots(figsize=(12, 5.5))

test_start = int(temporal_test["plot_year_int"].min())
test_end = int(temporal_test["plot_year_int"].max())

ax.axvspan(test_start, test_end, alpha=0.12, label="Период прогноза")

ax.plot(
    temporal_plot["plot_year_int"],
    temporal_plot["actual_median"],
    marker="o",
    linewidth=2,
    label="Факт: медиана по году",
)

ax.plot(
    temporal_plot["plot_year_int"],
    temporal_plot["predicted_median"],
    marker="s",
    linestyle="--",
    linewidth=2,
    label="Прогноз нейросети: медиана по году",
)

ax.axvline(test_start, linestyle=":", linewidth=2)

ax.set_title("Временной ряд фактических значений и прогноз нейросетевой модели")
ax.set_xlabel("Год окончания интервала наблюдения")
ax.set_ylabel("Интенсивность изменения бровки, м/год")
ax.grid(alpha=0.3)
ax.legend(loc="best")

metrics_text = (
    f"MAE = {temporal_mae:.3f}\n"
    f"RMSE = {temporal_rmse:.3f}\n"
    f"R² = {temporal_r2:.3f}"
)

ax.text(
    0.02,
    0.95,
    metrics_text,
    transform=ax.transAxes,
    verticalalignment="top",
    bbox=dict(boxstyle="round", alpha=0.15),
)

output_path = reports_dir / "figures" / "03_temporal_log_mlp_actual_vs_predicted.png"
fig.savefig(output_path, dpi=240, bbox_inches="tight", facecolor="white")

plt.show()

print("Сохранено:", output_path)
print("Сохранено:", temporal_metrics_path)
print("Сохранено:", temporal_predictions_path)


### Крупные ошибки нейросетевой модели

Таблица ниже показывает наблюдения, на которых нейросетевая модель ошиблась сильнее всего. Эти строки не удаляются из данных. Они нужны для интерпретации ограничений модели и для дальнейшей ручной проверки.


In [ ]:
log_predictions.sort_values("absolute_error", ascending=False).head(10)


## Групповая проверка с разделением по профилям

Случайное разбиение может распределить интервалы одних и тех же профилей между обучающей и тестовой выборками. Поэтому отдельно сохранена групповая проверка с разделением по `profile_id`: тестовая выборка содержит профили, которые не использовались при обучении.

Эта таблица не заменяет случайное разбиение, а показывает более строгую проверку обобщения. Отрицательный R2 — плохой сигнал для обобщающей способности модели: на такой проверке модель может быть хуже простой опоры, даже если MAE выглядит приемлемо.

In [ ]:
group_display = group_metrics.copy()
for column in ['MAE', 'RMSE', 'R2']:
    group_display[column] = group_display[column].round(4)
group_display

## Анализ ошибок прогноза

Для модели, лучшей по MAE на тестовой выборке при данном случайном разбиении, сохранены наблюдения с наибольшими ошибками прогноза. В текущем прогоне разрыв между MAE и RMSE действительно связан с несколькими крупными промахами: первые строки `baseline_worst_predictions.csv` имеют абсолютные ошибки порядка десятков м/год.

Это не повод удалять строки автоматически. Таблица нужна для ручной проверки: какие интервалы дают наибольший вклад в ошибку и связаны ли они с особенностями исходных наблюдений.

In [ ]:
worst_predictions.head(15)

In [ ]:
display(Image(filename=str(reports_dir / 'figures' / 'baseline_residuals.png')))

## Важность признаков

Для модели, лучшей по MAE на тестовой выборке при данном случайном разбиении, считается перестановочная важность признаков. Значение показывает, насколько растёт MAE, если перемешать один исходный признак.

Перестановочная важность показывает вклад признаков в текущей модели и текущей схеме проверки качества. Это не причинная интерпретация, и оценка может быть нестабильной при коррелированных признаках.

In [ ]:
feature_importance.head(15)

In [ ]:
display(Image(filename=str(reports_dir / 'figures' / 'baseline_feature_importance.png')))

## Ограничения интерпретации

Результаты базового моделирования нужно читать осторожно.

Во-первых, модели проверяют предсказательную способность признаков, а не причинные связи. Хорошее качество модели не доказывает, что конкретный признак физически вызывает изменение береговой бровки.

Во-вторых, исходные данные имеют ограничения: неодинаковое покрытие по участкам, пропуски, конфликтующие наблюдения и неполные сведения по внешним факторам.

В-третьих, ветровые данные нельзя использовать как сильный объясняющий фактор без проверки покрытия по интервалам наблюдений.

В-четвёртых, водный фактор в текущем виде является скорее годовым контекстом, а не локальным датированным рядом для каждого участка.

В-пятых, наблюдения с большими ошибками прогноза нельзя автоматически удалять. Их нужно рассматривать как кандидатов на ручную проверку.

Поэтому базовое моделирование используется как диагностический и сравнительный этап, а не как окончательная физико-экологическая модель.


## Выводы по базовому моделированию

В этом ноутбуке построена базовая постановка задачи регрессии для подготовленного датасета.

Были сравнены простая опорная модель, линейная модель, ансамблевые модели и нейросетевая модель `MLPRegressor`. Отдельно показано сопоставление фактических значений и прогнозов нейросетевой модели с логарифмическим преобразованием целевой переменной.

Основной вывод состоит в том, что в данных есть ограниченный предсказательный сигнал, но качество моделей существенно зависит от структуры и качества исходных наблюдений. Нейросетевая модель может использоваться как один из методов статистической обработки, однако на данном датасете она не является окончательным высокоточным решением.

Следующий ноутбук развивает идею нейросетевой диагностики: автоэнкодер используется не для прямого прогноза, а для поиска нетипичных строк, которые могут указывать на ошибки, выбросы или необычные сочетания признаков.
